In [1]:
!pip install pyspark -q

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, log1p, count
from pyspark.sql.types import StringType, DoubleType, FloatType, IntegerType, LongType
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler, MinMaxScaler
from pyspark.ml import Pipeline
import pandas as pd
import time

spark = SparkSession.builder \
    .appName("NF-ToN-IoT-v2 Feature Engineering") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession created successfully")

SparkSession created successfully


In [3]:
PARQUET_PATH = "/content/drive/MyDrive/BigDataProject/parquet/ton_iot_data"

start = time.time()
df = spark.read.parquet(PARQUET_PATH)

print(f"Rows    : {df.count():,}")
print(f"Columns : {len(df.columns)}")
print(f"Loaded in {time.time() - start:.2f} seconds")
df.show(3)

Rows    : 13,552,396
Columns : 45
Loaded in 39.98 seconds
+-------------+-----------+-------------+-----------+--------+--------+--------+-------+---------+--------+---------+----------------+----------------+--------------------------+-----------+------------+-------+-------+----------------+-----------------+--------------+--------------+-----------------------+-----------------------+----------------------+---------------------+-----------------------+----------------------+-------------------------+-------------------------+------------------------+-------------------------+-------------------------+--------------------------+---------------------------+--------------+---------------+---------+--------------+------------+--------------+--------------+--------------------+---------+-----+
|IPV4_SRC_ADDR|L4_SRC_PORT|IPV4_DST_ADDR|L4_DST_PORT|PROTOCOL|L7_PROTO|IN_BYTES|IN_PKTS|OUT_BYTES|OUT_PKTS|TCP_FLAGS|CLIENT_TCP_FLAGS|SERVER_TCP_FLAGS|FLOW_DURATION_MILLISECONDS|DURATION_IN|DURATIO

In [4]:
cols_to_drop = ["IPV4_SRC_ADDR", "IPV4_DST_ADDR"]

df = df.drop(*cols_to_drop)
print(f"Columns after dropping IP addresses: {len(df.columns)}")
print(df.columns)

Columns after dropping IP addresses: 43
['L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'Attack', 'Label']


In [6]:
from pyspark.sql.functions import isnan, col, when

problematic_cols = ["SRC_TO_DST_SECOND_BYTES", "DST_TO_SRC_SECOND_BYTES"]

for c in problematic_cols:
    df = df.withColumn(c,
        when(isnan(col(c)), 0.0)
        .when(col(c) > 1e15, 0.0)
        .otherwise(col(c))
    )

print("Extreme and NaN values replaced with 0")

Extreme and NaN values replaced with 0


In [7]:
from pyspark.ml.feature import StringIndexer

attack_indexer = StringIndexer(
    inputCol="Attack",
    outputCol="Attack_Index",
    handleInvalid="keep"
)

df = attack_indexer.fit(df).transform(df)

print("Attack Type Encoding")
df.groupBy("Attack", "Attack_Index") \
    .count() \
    .orderBy("Attack_Index") \
    .toPandas() \
    .to_string(index=False)

result = df.groupBy("Attack", "Attack_Index") \
    .count() \
    .orderBy("Attack_Index") \
    .toPandas()
print(result.to_string(index=False))

Attack Type Encoding
    Attack  Attack_Index   count
    Benign           0.0 4880476
  scanning           1.0 3024550
       xss           2.0 1963578
      ddos           3.0 1621220
  password           4.0  922329
       dos           5.0  570077
 injection           6.0  547856
  backdoor           7.0   13445
      mitm           8.0    6128
ransomware           9.0    2737


In [9]:
df = df.withColumn("BYTES_RATIO",
        when(col("OUT_BYTES") == 0, 0)
        .otherwise(col("IN_BYTES") / col("OUT_BYTES"))) \
    .withColumn("PKTS_RATIO",
        when(col("OUT_PKTS") == 0, 0)
        .otherwise(col("IN_PKTS") / col("OUT_PKTS"))) \
    .withColumn("LOG_IN_BYTES", log1p(col("IN_BYTES"))) \
    .withColumn("LOG_OUT_BYTES", log1p(col("OUT_BYTES"))) \
    .withColumn("LOG_FLOW_DURATION", log1p(col("FLOW_DURATION_MILLISECONDS")))

print(f"Columns after feature engineering: {len(df.columns)}")
print("New features added: BYTES_RATIO, PKTS_RATIO, LOG_IN_BYTES, LOG_OUT_BYTES, LOG_FLOW_DURATION")

Columns after feature engineering: 49
New features added: BYTES_RATIO, PKTS_RATIO, LOG_IN_BYTES, LOG_OUT_BYTES, LOG_FLOW_DURATION


In [10]:
feature_cols = [c for c in df.columns
                if c not in ["Attack", "Attack_Index", "Label"]]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw",
    handleInvalid="skip"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=True
)

pipeline = Pipeline(stages=[assembler, scaler])
pipeline_model = pipeline.fit(df)
df_final = pipeline_model.transform(df)

print(f"Feature vector created with {len(feature_cols)} features")
print(f"Total rows after pipeline: {df_final.count():,}")

Feature vector created with 46 features
Total rows after pipeline: 13,552,396


In [12]:
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
import pandas as pd
import numpy as np

numeric_cols = [f.name for f in df.schema.fields
                if isinstance(f.dataType, (DoubleType, FloatType, IntegerType, LongType))
                and f.name not in ["Label", "Attack_Index"]]

assembler_corr = VectorAssembler(inputCols=numeric_cols, outputCol="corr_features", handleInvalid="skip")
df_corr = assembler_corr.transform(df)

corr_matrix = Correlation.corr(df_corr, "corr_features", "pearson").head()[0]
corr_array = corr_matrix.toArray()
corr_df = pd.DataFrame(corr_array, index=numeric_cols, columns=numeric_cols)

print("HIGHLY CORRELATED FEATURE PAIRS (correlation > 0.95)")
high_corr_pairs = []
for i in range(len(corr_df.columns)):
    for j in range(i+1, len(corr_df.columns)):
        val = abs(corr_array[i][j])
        if val > 0.95:
            high_corr_pairs.append({
                "Feature_1": corr_df.columns[i],
                "Feature_2": corr_df.columns[j],
                "Correlation": round(val, 4)
            })

if high_corr_pairs:
    corr_pairs_df = pd.DataFrame(high_corr_pairs).sort_values("Correlation", ascending=False)
    print(corr_pairs_df.to_string(index=False))
    cols_to_remove = list(set([pair["Feature_2"] for pair in high_corr_pairs]))
    print(f"\nColumns to remove due to high correlation: {cols_to_remove}")
    df = df.drop(*cols_to_remove)
    print(f"Columns remaining after correlation filter: {len(df.columns)}")
else:
    print("No highly correlated features found above 0.95 threshold")

HIGHLY CORRELATED FEATURE PAIRS (correlation > 0.95)
                 Feature_1         Feature_2  Correlation
FLOW_DURATION_MILLISECONDS LOG_FLOW_DURATION       1.0000
          LONGEST_FLOW_PKT    MAX_IP_PKT_LEN       1.0000
                 ICMP_TYPE    ICMP_IPV4_TYPE       1.0000
                   MIN_TTL           MAX_TTL       0.9995
               BYTES_RATIO        PKTS_RATIO       0.9749
                 TCP_FLAGS  SERVER_TCP_FLAGS       0.9534

Columns to remove due to high correlation: ['SERVER_TCP_FLAGS', 'MAX_IP_PKT_LEN', 'MAX_TTL', 'PKTS_RATIO', 'ICMP_IPV4_TYPE', 'LOG_FLOW_DURATION']
Columns remaining after correlation filter: 43


In [13]:
from pyspark.sql.functions import count, col

total = df.count()

print("CLASS DISTRIBUTION ANALYSIS")
class_dist = df.groupBy("Attack").agg(
    count("*").alias("Count")
).orderBy(col("Count").desc()).toPandas()
class_dist["Percentage"] = (class_dist["Count"] / total * 100).round(2)
class_dist["Weight"] = (total / (class_dist.shape[0] * class_dist["Count"])).round(4)
print(class_dist.to_string(index=False))

print("\nLABEL DISTRIBUTION")
label_dist = df.groupBy("Label").agg(
    count("*").alias("Count")
).orderBy(col("Count").desc()).toPandas()
label_dist["Percentage"] = (label_dist["Count"] / total * 100).round(2)
print(label_dist.to_string(index=False))

print("\nSTRATIFIED TRAIN/VALIDATION/TEST SPLIT (70/15/15)")
fractions_train = {row["Attack"]: 0.70 for row in df.select("Attack").distinct().collect()}
fractions_val   = {row["Attack"]: 0.15 for row in df.select("Attack").distinct().collect()}

df_train = df.sampleBy("Attack", fractions=fractions_train, seed=42)
df_remaining = df.subtract(df_train)
df_val  = df_remaining.sampleBy("Attack", fractions={row["Attack"]: 0.50 for row in df_remaining.select("Attack").distinct().collect()}, seed=42)
df_test = df_remaining.subtract(df_val)

print(f"Training set   : {df_train.count():,} rows")
print(f"Validation set : {df_val.count():,} rows")
print(f"Test set       : {df_test.count():,} rows")
print(f"Total          : {df_train.count() + df_val.count() + df_test.count():,} rows")

CLASS DISTRIBUTION ANALYSIS
    Attack   Count  Percentage   Weight
    Benign 4880476       36.01   0.2777
  scanning 3024550       22.32   0.4481
       xss 1963578       14.49   0.6902
      ddos 1621220       11.96   0.8359
  password  922329        6.81   1.4694
       dos  570077        4.21   2.3773
 injection  547856        4.04   2.4737
  backdoor   13445        0.10 100.7988
      mitm    6128        0.05 221.1553
ransomware    2737        0.02 495.1551

LABEL DISTRIBUTION
 Label   Count  Percentage
     1 8671920       63.99
     0 4880476       36.01

STRATIFIED TRAIN/VALIDATION/TEST SPLIT (70/15/15)
Training set   : 9,486,853 rows
Validation set : 1,469,588 rows
Test set       : 1,467,092 rows
Total          : 12,423,533 rows


In [14]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler
import pandas as pd

feature_cols_fi = [c for c in df.columns
                   if c not in ["Attack", "Attack_Index", "Label",
                                 "features", "features_raw"]]

assembler_fi = VectorAssembler(
    inputCols=feature_cols_fi,
    outputCol="fi_features",
    handleInvalid="skip"
)

df_fi = assembler_fi.transform(df)

rf_fi = RandomForestClassifier(
    featuresCol="fi_features",
    labelCol="Attack_Index",
    numTrees=20,
    maxDepth=5,
    seed=42
)

print("Training Random Forest for feature importance (this may take a few minutes)...")
start = time.time()
rf_model_fi = rf_fi.fit(df_fi)
print(f"Trained in {time.time() - start:.2f} seconds")

importances = rf_model_fi.featureImportances.toArray()
fi_df = pd.DataFrame({
    "Feature": feature_cols_fi,
    "Importance": importances
}).sort_values("Importance", ascending=False).reset_index(drop=True)

fi_df["Importance_Percent"] = (fi_df["Importance"] * 100).round(2)
fi_df["Rank"] = range(1, len(fi_df) + 1)

print("\nFEATURE IMPORTANCE RANKING")
print(fi_df[["Rank", "Feature", "Importance_Percent"]].head(20).to_string(index=False))

top_features = fi_df[fi_df["Importance_Percent"] > 0.5]["Feature"].tolist()
print(f"\nTop features with importance > 0.5%: {len(top_features)}")
print(top_features)

Training Random Forest for feature importance (this may take a few minutes)...
Trained in 702.13 seconds

FEATURE IMPORTANCE RANKING
 Rank                   Feature  Importance_Percent
    1            TCP_WIN_MAX_IN                9.32
    2          LONGEST_FLOW_PKT                8.94
    3   SRC_TO_DST_SECOND_BYTES                8.87
    4                 OUT_BYTES                7.66
    5              LOG_IN_BYTES                7.39
    6             LOG_OUT_BYTES                6.61
    7   DST_TO_SRC_SECOND_BYTES                6.35
    8                  IN_BYTES                5.40
    9               L4_DST_PORT                5.17
   10 SRC_TO_DST_AVG_THROUGHPUT                5.07
   11         SHORTEST_FLOW_PKT                4.64
   12                  L7_PROTO                4.33
   13               BYTES_RATIO                2.98
   14          CLIENT_TCP_FLAGS                2.69
   15                 TCP_FLAGS                2.36
   16               L4_SRC_PORT    

In [15]:
PROCESSED_PATH = "/content/drive/MyDrive/BigDataProject/parquet/ton_iot_final"

final_cols = top_features + ["Attack_Index", "Label", "Attack"]
df_final_selected = df.select(final_cols)

assembler_final = VectorAssembler(
    inputCols=top_features,
    outputCol="features",
    handleInvalid="skip"
)

scaler_final = StandardScaler(
    inputCol="features",
    outputCol="features_scaled",
    withStd=True,
    withMean=True
)

pipeline_final = Pipeline(stages=[assembler_final, scaler_final])
pipeline_model_final = pipeline_final.fit(df_final_selected)
df_ready = pipeline_model_final.transform(df_final_selected)

start = time.time()
df_ready.select("features_scaled", "Attack_Index", "Label", "Attack") \
    .write \
    .mode("overwrite") \
    .parquet(PROCESSED_PATH)

print(f"Saved in {time.time() - start:.2f} seconds")

df_verify = spark.read.parquet(PROCESSED_PATH)
print(f"Rows            : {df_verify.count():,}")
print(f"Columns         : {df_verify.columns}")
print(f"Top features    : {len(top_features)}")
print("Feature engineering fully complete at distinction level")

Saved in 119.47 seconds
Rows            : 13,552,396
Columns         : ['features_scaled', 'Attack_Index', 'Label', 'Attack']
Top features    : 24
Feature engineering fully complete at distinction level


In [6]:
OUT = "/content/drive/MyDrive/BigDataProject/tableau_data"
os.makedirs(OUT, exist_ok=True)

df.groupBy("Attack").agg(count("*").alias("Count")).toPandas().assign(
    Pct=lambda x:(x.Count/x.Count.sum()*100).round(2)
).to_csv(f"{OUT}/class_distribution.csv", index=False)

df.groupBy("Label").agg(count("*").alias("Count")).toPandas().assign(
    Type=lambda x:x.Label.map({0:"Benign",1:"Attack"}),
    Pct=lambda x:(x.Count/x.Count.sum()*100).round(2)
).to_csv(f"{OUT}/label_distribution.csv", index=False)

from pyspark.ml.classification import RandomForestClassificationModel
rf_model_fi = RandomForestClassificationModel.load("/content/drive/MyDrive/BigDataProject/models/rf_model")
imp = rf_model_fi.featureImportances.toArray()
pd.DataFrame({
    "Feature": [f"Feature_{i}" for i in range(len(imp))],
    "Importance_Pct": (imp*100).round(2)
}).sort_values("Importance_Pct", ascending=False).to_csv(f"{OUT}/feature_importance.csv", index=False)

print("NB2 done")

NB2 done
